In [186]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
import os
from typing import Final

In [187]:
%cd ..
from src.preprocessing.coordinate_transformer import CoordinateTransformater

/


/home/dascim/miniconda3/envs/histograph/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [188]:
patient = "002"
REFERENCE_POINTS: Final[dict[str, dict[str, tuple[float, float]]]] = {
    "001": {
        "origin_point": (16038.24, 69265.17),
        "target_point": (380, 1001)
    },
    "002": {
        "origin_point": (20212.06, 59714.98),
        "target_point": (632, 1261)
    }
}

In [189]:
def clean_indices(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Resets indices of a DataFrame and gives a warning, if any indices where missing
    :param df: DataFrame
    :param name: Name of the data, that is checked
    :return: DataFrame with cleaned indices
    """
    prior_max_index = df.index.max()
    df.reset_index(drop=True, inplace=True)
    posterior_max_index = df.index.max()

    if prior_max_index != posterior_max_index:
        print(
            f"[RAW_DATA_GENERATION] - Warning: {prior_max_index - posterior_max_index} indices were missing and have been reset in {name}.")

    return df

# Matchings across 25 and 106

In [190]:
df_matchings_2 = np.load('/home/dascim/repos/from_odyssee/resultats/matching_across_stainings_002_from_segmentation.npy', allow_pickle=True)
df_matchings_2 = clean_indices(df_matchings_2, 'matchings2')
df_matchings_2.head()

,s1,s2
0,1,4
1,2,1
2,3,3
3,4,2
4,6,6


In [191]:
df_centroids_2 = pd.read_pickle('/home/dascim/repos/from_odyssee/resultats/IFTA_EXC_002_25_centroids_from_segmentation.npy')
df_centroids_2 = clean_indices(df_centroids_2, 'centroids2')
df_centroids_2.head()

[RAW_DATA_GENERATION] - Warning: 1 indices were missing and have been reset in centroids2.


,centroid,label gt,centroid non register
0,"[510, 1165]",1,"[510, 1165]"
1,"[632, 1261]",4,"[632, 1261]"
2,"[664, 1055]",2,"[664, 1055]"
3,"[672, 1225]",3,"[672, 1225]"
4,"[759, 1296]",6,"[759, 1296]"


In [192]:
# Get target df
df_targets = pd.read_csv('/home/dascim/repos/histograph/data/input/annotations_exc.csv', delimiter=';')
df_targets = df_targets[['Center X', 'Center Y', 'Term', 'Image filename']]

# Get staining and patient from image filename
df_targets["patient"] = df_targets['Image filename'].str.split('_').str[2]
df_targets['staining'] = df_targets['Image filename'].str.split('_').str[-1]
df_targets['staining'] = df_targets['staining'].str.split('.').str[0]
df_targets.drop(columns=['Image filename'], inplace=True)

# Filter by patient and staining
df_targets = df_targets[(df_targets['patient'] == patient) & (df_targets['staining'] == "25")]


In [193]:
df_2 = pd.merge(df_matchings_2, df_centroids_2, left_on="s1", right_index=True, how="left").dropna()
df_2

,s1,s2,centroid,label gt,centroid non register
0,1,4,"[632, 1261]",4,"[632, 1261]"
1,2,1,"[664, 1055]",2,"[664, 1055]"
2,3,3,"[672, 1225]",3,"[672, 1225]"
3,4,2,"[759, 1296]",6,"[759, 1296]"
4,6,6,"[828, 1306]",8,"[828, 1306]"
...,...,...,...,...,...
110,153,138,"[3619, 3526]",228,"[3619, 3526]"
111,154,139,"[3672, 3261]",197,"[3672, 3261]"
112,155,141,"[3721, 3394]",196,"[3721, 3394]"
113,157,142,"[3929, 3065]",181,"[3929, 3065]"


In [194]:
# Extract centroid coordinates
df_2['centroid_x'] = df_2["centroid"].apply(lambda a: a[0])
df_2['centroid_y'] = df_2["centroid"].apply(lambda a: a[1])
df_2

,s1,s2,centroid,label gt,centroid non register,centroid_x,centroid_y
0,1,4,"[632, 1261]",4,"[632, 1261]",632,1261
1,2,1,"[664, 1055]",2,"[664, 1055]",664,1055
2,3,3,"[672, 1225]",3,"[672, 1225]",672,1225
3,4,2,"[759, 1296]",6,"[759, 1296]",759,1296
4,6,6,"[828, 1306]",8,"[828, 1306]",828,1306
...,...,...,...,...,...,...,...
110,153,138,"[3619, 3526]",228,"[3619, 3526]",3619,3526
111,154,139,"[3672, 3261]",197,"[3672, 3261]",3672,3261
112,155,141,"[3721, 3394]",196,"[3721, 3394]",3721,3394
113,157,142,"[3929, 3065]",181,"[3929, 3065]",3929,3065


In [195]:
coord_transformer = CoordinateTransformater(
    rotation= 90,
    mirror_x= False ,
    mirror_y= False ,
    magnification= 0.0625
)
coord_transformer.calculate_offset(**REFERENCE_POINTS['002'])
transformation_results = coord_transformer.match_coordinates(df_targets[['Center X', 'Center Y']].to_numpy(),
                                                                     df_2[['centroid_x', 'centroid_y']].to_numpy())
# Add match index to df_targets
df_targets_2 = df_targets.copy()
df_targets_2['match_index'] = np.array(transformation_results)[:, 0]

# Merge targets into df wit determined indeces
df_2 = pd.merge(df_2, df_targets_2, left_index=True, right_on='match_index', how='left')

[COORD_TRANSFORM] - Calculated offset: (4364, 0)


In [196]:
df_2

,s1,s2,centroid,label gt,centroid non register,centroid_x,centroid_y,Center X,Center Y,Term,patient,staining,match_index
2700.0,1,4,"[632, 1261]",4,"[632, 1261]",632,1261,20212.06,59714.98,Healthy,002,25,0.0
2699.0,2,1,"[664, 1055]",2,"[664, 1055]",664,1055,16909.14,59172.29,Healthy,002,25,1.0
2698.0,3,3,"[672, 1225]",3,"[672, 1225]",672,1225,19585.02,59115.48,Healthy,002,25,2.0
2697.0,4,2,"[759, 1296]",6,"[759, 1296]",759,1296,20744.04,57652.53,Healthy,002,25,3.0
2696.0,6,6,"[828, 1306]",8,"[828, 1306]",828,1306,20920.42,56574.65,Healthy,002,25,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2473.0,153,138,"[3619, 3526]",228,"[3619, 3526]",3619,3526,56461.14,11874.03,Healthy,002,25,110.0
2469.0,154,139,"[3672, 3261]",197,"[3672, 3261]",3672,3261,52190.46,11061.49,Healthy,002,25,111.0
2465.0,155,141,"[3721, 3394]",196,"[3721, 3394]",3721,3394,54347.00,10336.05,Healthy,002,25,112.0
2452.0,157,142,"[3929, 3065]",181,"[3929, 3065]",3929,3065,49037.54,6910.45,Healthy,002,25,113.0


In [197]:
df_2[df_2.isna().any(axis=1)]

,s1,s2,centroid,label gt,centroid non register,centroid_x,centroid_y,Center X,Center Y,Term,patient,staining,match_index
NaN,14,15,"[1076, 972]",-1,"[1076, 972]",1076,972,NaN,NaN,NaN,NaN,NaN,10.0
NaN,56,58,"[1776, 875]",-1,"[1776, 875]",1776,875,NaN,NaN,NaN,NaN,NaN,40.0
NaN,124,115,"[3257, 1637]",-1,"[3257, 1637]",3257,1637,NaN,NaN,NaN,NaN,NaN,89.0


In [198]:
df_2.groupby("Term")['Term'].value_counts()

Term
Dead         1
Healthy    111
Name: count, dtype: int64

# Matching across all stainings

In [199]:
df_matchings_all = pd.read_pickle('/home/dascim/repos/from_odyssee/resultats_all_stainings/matching_across_stainings_002_from_segmentation.npy')
df_matchings_all = clean_indices(df_matchings_all, 'matchings_all')
df_matchings_all.head()

[RAW_DATA_GENERATION] - Warning: 16 indices were missing and have been reset in matchings_all.


,s1,s2,s3,s4
0,2,1,3,4
1,3,3,2,2
2,4,2,4,5
3,6,6,8,12
4,7,7,9,14


In [200]:
df_centroids_all = pd.read_pickle('/home/dascim/repos/from_odyssee/resultats_all_stainings/IFTA_EXC_002_25_centroids_from_segmentation.npy')
df_centroids_all = clean_indices(df_centroids_all, 'centroids_all')
df_centroids_all.head()

[RAW_DATA_GENERATION] - Warning: 1 indices were missing and have been reset in centroids_all.


,centroid,label gt,centroid non register
0,"[510, 1165]",1,"[510, 1165]"
1,"[632, 1261]",4,"[632, 1261]"
2,"[664, 1055]",2,"[664, 1055]"
3,"[672, 1225]",3,"[672, 1225]"
4,"[759, 1296]",6,"[759, 1296]"


In [201]:
df_all = pd.merge(df_matchings_all, df_centroids_all, left_on="s1", right_index=True, how="left").dropna()
# Extract centroid coordinates
df_all['centroid_x'] = df_all["centroid"].apply(lambda a: a[0])
df_all['centroid_y'] = df_all["centroid"].apply(lambda a: a[1])
df_all

,s1,s2,s3,s4,centroid,label gt,centroid non register,centroid_x,centroid_y
0,2,1,3,4,"[664, 1055]",2.0,"[664, 1055]",664,1055
1,3,3,2,2,"[672, 1225]",3.0,"[672, 1225]",672,1225
2,4,2,4,5,"[759, 1296]",6.0,"[759, 1296]",759,1296
3,6,6,8,12,"[828, 1306]",8.0,"[828, 1306]",828,1306
4,7,7,9,14,"[835, 1376]",9.0,"[835, 1376]",835,1376
...,...,...,...,...,...,...,...,...,...
95,151,137,156,136,"[3601, 3159]",198.0,"[3601, 3159]",3601,3159
96,152,140,157,137,"[3607, 3430]",199.0,"[3607, 3430]",3607,3430
97,153,138,161,142,"[3619, 3526]",228.0,"[3619, 3526]",3619,3526
98,154,139,162,146,"[3672, 3261]",197.0,"[3672, 3261]",3672,3261


In [202]:
coord_transformer = CoordinateTransformater(
    rotation= 90,
    mirror_x= False ,
    mirror_y= False ,
    magnification= 0.0625
)
coord_transformer.calculate_offset(**REFERENCE_POINTS['002'])
transformation_results = coord_transformer.match_coordinates(df_targets[['Center X', 'Center Y']].to_numpy(),
                                                                     df_all[['centroid_x', 'centroid_y']].to_numpy())
# Add match index to df_targets
df_targets_all = df_targets.copy()
df_targets_all['match_index'] = np.array(transformation_results)[:, 0]

# Merge targets into df wit determined indeces
df_all = pd.merge(df_all, df_targets_2, left_index=True, right_on='match_index', how='left')

[COORD_TRANSFORM] - Calculated offset: (4364, 0)


In [203]:
df_all.groupby("Term")['Term'].value_counts()

Term
Healthy    97
Name: count, dtype: int64

# Only Staining 25 without matchings

In [204]:
df_centroids_25 = df_centroids_2.copy()
df_centroids_25['centroid_x'] = df_centroids_25["centroid"].apply(lambda a: a[0])
df_centroids_25['centroid_y'] = df_centroids_25["centroid"].apply(lambda a: a[1])

In [205]:
coord_transformer = CoordinateTransformater(
    rotation= 90,
    mirror_x= False ,
    mirror_y= False ,
    magnification= 0.0625
)
coord_transformer.calculate_offset(**REFERENCE_POINTS['002'])
transformation_results = coord_transformer.match_coordinates(df_targets[['Center X', 'Center Y']].to_numpy(),
                                                                     df_centroids_25[['centroid_x', 'centroid_y']].to_numpy())
# Add match index to df_targets
df_targets_25 = df_targets.copy()
df_targets_25['match_index'] = np.array(transformation_results)[:, 0]

# Merge targets into df wit determined indeces
df_centroids_25 = pd.merge(df_centroids_25, df_targets_2, left_index=True, right_on='match_index', how='left')

[COORD_TRANSFORM] - Calculated offset: (4364, 0)


In [206]:
df_centroids_25.groupby("Term")['Term'].value_counts()

Term
Dead         1
Healthy    111
Name: count, dtype: int64

## Annotations

In [214]:
df_annotations = pd.read_csv('/home/dascim/repos/histograph/data/input/annotations_exc.csv', delimiter=';')
# Get staining and patient from image filename
df_annotations["patient"] = df_annotations['Image filename'].str.split('_').str[2]
df_annotations['staining'] = df_annotations['Image filename'].str.split('_').str[-1]
df_annotations['staining'] = df_annotations['staining'].str.split('.').str[0]

In [221]:
df_annotations[(df_annotations['Term']=='Dead') & (df_annotations['patient']=='002') & (df_annotations['staining']=='25')]

,ID,Area (microns²),Perimeter (mm),Center X,Center Y,Image ID,Image filename,User,Term,Annotation thumb,Annotation in Cytomine,patient,staining
2419,1027540,13738.183375,0.450717,57613.28,3586.52,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2420,1027527,16052.214894,0.491115,63066.17,4903.12,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2421,1027514,8211.898152,0.341286,60109.72,4891.94,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2422,1027501,8616.251155,0.350150,56661.36,10557.91,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2423,1027488,8359.561595,0.346919,57743.43,11307.71,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2424,1027475,6234.508679,0.311129,36084.13,13086.23,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2425,1027462,11465.977081,0.400312,54981.53,15088.77,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2426,1027449,8819.384026,0.356429,47489.71,15533.13,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2427,1027436,6839.061687,0.312857,24305.99,22533.48,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25
2428,1027423,8264.944777,0.345429,25954.70,22866.45,21507,IFTA_EXC_002_NX_NE4_25.svs,lampert,Dead,http://cytomine.icube.unistra.fr/api/annotatio...,http://cytomine.icube.unistra.fr/#/project/140...,002,25


In [224]:
df_annotations[(df_annotations['Term']=='Dead') & (df_annotations['patient']=='002') & (df_annotations['staining']=='25')]['Annotation in Cytomine'][2419]

'http://cytomine.icube.unistra.fr/#/project/1407/image/21507/annotation/1027540'